#### Prepare the tables. We use SALES_REFUNDS dataset and split it into train and test sets.

In [1]:
# First we prepare the data for the scenario.
import certifi
import pandas as pd
from hana_ml import dataframe
sslstore = certifi.where()

cc = dataframe.ConnectionContext(userkey="RaysKey", sslValidateCertificate=False, encrypt=True, sslKeyStore=sslstore)

ts_df = dataframe.create_dataframe_from_pandas(cc, pandas_df=pd.read_csv("./SALES_REFUNDS.csv"), table_name="SALES_REFUNDS", force=True).to_datetime({"BOOKING_DATE":"YYYY-MM-DD"}).sort_values(by="BOOKING_DATE").smart_save("SALES_REFUNDS", force=True)

/Users/I308290/Projects/hanamlapi/src/hana_ml/_compat/pandas_compat.py:100: RuntimeWarning: Detected pandas 3.0.1. Note: string columns may use ExtensionArray and Copy-on-Write semantics are enabled by default. hana_ml aims to be compatible but if you encounter issues with .values on Series, prefer .to_numpy()/.array.
  warnings.warn(
100%|██████████| 1/1 [00:00<00:00,  2.64it/s]


#### Setup Chatbot with HANA ML Toolkit

In [2]:
from gen_ai_hub.proxy.langchain import init_llm
from hana_ai.agents.mem0_context_chat_agent import Mem0ContextChatAgent
from hana_ai.tools.toolkit import HANAMLToolkit
import os, certifi
os.environ["SSL_CERT_FILE"] = certifi.where()
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()

llm = init_llm('gpt-4.1', temperature=0.0, max_tokens=1800)
tools = HANAMLToolkit(cc, used_tools='all').get_tools()
chatbot = Mem0ContextChatAgent(llm=llm, tools=tools, verbose=True)
chatbot.clear_long_term_memory()

/Users/I308290/.pyenv/versions/3.12.0/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
for tool in tools:
    print(f"tool name: {tool.name}.")
    print(f"tool description: {tool.description}")
    print()

tool name: accuracy_measure.
tool description: To compute the accuracy measure using true and predict tables.

tool name: additive_model_forecast_fit_and_save.
tool description: To fit the additive model forecast and save the model in model storage.

tool name: additive_model_forecast_load_model_and_predict.
tool description: To load the additive model forecast from model storage and predict.

tool name: automatic_timeseries_fit_and_save.
tool description: To fit an AutomaticTimeseries model and save it in the model storage.

tool name: automatic_timeseries_load_model_and_predict.
tool description: To load a model and do the prediction using automatic timeseries model.

tool name: automatic_timeseries_load_model_and_score.
tool description: To load a model and do the scoring for automatic timeseries.

tool name: cap_artifacts.
tool description: To generate CAP artifacts for a given model from model storage. 

tool name: delete_models.
tool description: Delete models from the model stor

##### Fetch the data. The tool `fetch_data` in the toolkit is used to fetch the data from the HANA database.

In [4]:
ts_df.head(150).save("SALES_REFUNDS_TRAIN", force=True)
ts_df.tail(30).select(ts_df.columns[0]).save("SALES_REFUNDS_PREDICT", force=True)

In [5]:
ts_df.head(10).collect()

,BOOKING_DATE,REFUNDS
0,2024-05-01,0.000000
1,2024-05-02,3917.179500
2,2024-05-03,1968.674500
3,2024-05-04,10501.144875
4,2024-05-05,0.000000
5,2024-05-06,67.290000
6,2024-05-07,903.660000
7,2024-05-08,2130.070000
8,2024-05-09,4104.270000
9,2024-05-10,11519.010000


In [4]:
print(chatbot.chat("Show me the last 10 records of data from SALES_REFUNDS"))


Here are the last 10 records from the SALES_REFUNDS table:

| BOOKING_DATE | REFUNDS   |
|--------------|-----------|
| 2024-10-19   | 2771.56   |
| 2024-10-20   | 2300.83   |
| 2024-10-21   | 2260.7    |
| 2024-10-22   | 0         |
| 2024-10-23   | 8436.4    |
| 2024-10-24   | 0         |
| 2024-10-25   | 168.3     |
| 2024-10-26   | 503.582   |
| 2024-10-27   | 6201.68   |
| 2024-10-28   | 0         |

Let me know if you need further analysis or details.


##### Create a dataset report for the given data using the tool `ts_dataset_report`

In [ ]:
chatbot.chat("Then create a dataset report for me on SALES_REFUNDS table")

 75%|███████▌  | 6/8 [00:49<00:22, 11.15s/it]

##### Use `ts_check` to calculate the statistics of the dataset.

In [ ]:
print(chatbot.chat("I want to check the time series data SALES_REFUNDS_TRAIN and suggest the predict model for me."))

Here is the analysis of your SALES_REFUNDS_TRAIN time series data:

- The data is stationary (no significant trend detected).
- There is some seasonality present with a period of 28 days (additive type).
- The proportion of zero values is 14%, so the data is not highly intermittent.

Based on these characteristics, the following models are suitable for forecasting:
1. Additive Model Forecast (good for stationary data with additive seasonality)
2. Automatic Time Series Forecast (can automatically select the best model for your data)

Would you like to proceed with fitting one of these models? If so, do you have a preference, or should I recommend the best option for you?


##### Model suggestion

##### Use suggested model to train the model via the tool `xxx_fit_and_save`. 

In [ ]:
print(chatbot.chat("Then please train the table using the suggested model and save as my_hana_ai_model"))

The additive model forecast is being trained on your SALES_REFUNDS_TRAIN table using a seasonality period of 28 days, as suggested by the time series check. The model will be saved as my_hana_ai_model.

Would you like to proceed with making predictions, evaluating the model, or need further analysis? Let me know your next step!


##### Predict on the model using `xxx_load_model_and_predict` tool.

In [10]:
print(chatbot.chat("I want to predict the SALES_REFUNDS_PREDICT using the trained model, key is BOOKING_DATE and endog is REFUNDS."))

The prediction for SALES_REFUNDS_PREDICT using your trained additive model (my_hana_ai_model) is being generated. The results will be saved in the table: PREDICT_RESULT_SALES_REFUNDS_PREDICT_my_hana_ai_model_4.

Would you like to view the prediction results, visualize them, or evaluate the model's accuracy? Let me know your next step!


In [11]:
print(chatbot.chat("show me all the predicted results"))

Here are the last 10 predicted results for SALES_REFUNDS_PREDICT from all versions of your trained additive model (my_hana_ai_model). The predictions are consistent across all versions:

| BOOKING_DATE | YHAT     | YHAT_LOWER | YHAT_UPPER |
|--------------|----------|------------|------------|
| 2024-10-19   | 4970.78  | 921.12     | 9118.59    |
| 2024-10-20   | 1497.10  | -2572.41   | 5449.40    |
| 2024-10-21   | 216.71   | -3859.43   | 4305.35    |
| 2024-10-22   | 3081.59  | -930.91    | 7435.52    |
| 2024-10-23   | 4944.61  | 764.89     | 9193.40    |
| 2024-10-24   | 6580.90  | 2372.66    | 10769.70   |
| 2024-10-25   | 5445.79  | 1392.70    | 9535.23    |
| 2024-10-26   | 4849.63  | 1002.00    | 8652.85    |
| 2024-10-27   | 1150.75  | -2800.91   | 5251.90    |
| 2024-10-28   | -29.59   | -3968.73   | 3847.94    |

- YHAT: Predicted value
- YHAT_LOWER: Lower confidence bound
- YHAT_UPPER: Upper confidence bound

Would you like to visualize these predictions, compare them with 

##### Plot the predictions using the tool `forecast_line_plot`

In [12]:
result = chatbot.chat("Generate the line plot on the predicted results table and compared with the actual table SALES_REFUNDS")


##### Generate CAP artifacts using the tool `cap_artifacts`

In [13]:
print(chatbot.chat("I want to generate CAP artifacts, the project name is my_project and output path is cap"))

The CAP artifacts have been generated successfully for your model (my_hana_ai_model, version 4).

- Project name: my_project
- Output directory: cap/my_project

You can now use these artifacts for integration with your CAP (Cloud Application Programming) project. Let me know if you need the files, further instructions, or help with deployment!
